# Compression-aware baseline for PlantCLEF 2026

This notebook provides an end-to-end baseline for training/fine-tuning on compressed 512x512 inputs and generating inference outputs with the same preprocessing path.

In [ ]:
from pathlib import Path

import mlflow
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from src.config.compression import CompressionConfig
from src.config.mlflow_init import init_mlflow
from src.data.compression import preprocess_pil_image
from src.models.compression import CompressAICodec

In [ ]:
cfg = CompressionConfig.from_env()
cfg.validate()
cfg.ensure_output_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(cfg)
print(f"device={device}")

In [ ]:
init_mlflow()
mlflow.set_experiment(cfg.mlflow_experiment_name)

## Data loading and compressed training dataset

In [ ]:
train_df = pd.read_csv(cfg.train_metadata_path, sep=";", dtype={"partner": str})
if cfg.max_train_samples > 0:
    train_df = train_df.sample(
        n=min(cfg.max_train_samples, len(train_df)), random_state=42
    )

species_ids = sorted(train_df["species_id"].astype(int).unique())
class_to_idx = {species_id: idx for idx, species_id in enumerate(species_ids)}
num_classes = len(class_to_idx)
print(f"training samples={len(train_df)} classes={num_classes}")

In [ ]:
class CompressedTrainingDataset(Dataset):
    def __init__(self, df, cfg, class_to_idx, codec):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.class_to_idx = class_to_idx
        self.codec = codec

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        species_id = int(row["species_id"])
        if "image_id" in row.index:
            image_name = f"{row['image_id']}.jpg"
        elif "image_name" in row.index:
            image_name = str(row["image_name"])
        else:
            raise KeyError("Metadata must contain 'image_id' or 'image_name'.")
        candidate_paths = [
            Path(self.cfg.train_image_root) / str(species_id) / image_name,
            Path(self.cfg.train_image_root) / image_name,
        ]
        if "organ" in row.index and pd.notna(row["organ"]):
            candidate_paths.append(
                Path(self.cfg.train_image_root) / str(row["organ"]) / image_name
            )
        image_path = next((p for p in candidate_paths if p.exists()), None)
        if image_path is None:
            raise FileNotFoundError(
                f"Image not found for row index {idx}: {image_name}"
            )
        image = Image.open(image_path).convert("RGB")
        tensor = preprocess_pil_image(
            image,
            image_size=self.cfg.image_size,
            jpeg_normalize_enabled=self.cfg.jpeg_normalize,
            jpeg_quality=self.cfg.jpeg_quality,
        )
        tensor = self.codec.encode_decode_with_cache(
            image_path=image_path,
            input_tensor=tensor,
            cache_dir=self.cfg.compressed_cache_dir,
        ).squeeze(0)
        label = self.class_to_idx[species_id]
        return tensor, label

In [ ]:
compression_device = cfg.compression_device
if compression_device.startswith("cuda"):
    compression_device = f"cuda:{cfg.compression_gpu_id}"
    if not torch.cuda.is_available():
        compression_device = "cpu"

loader_workers = cfg.num_workers
if (
    cfg.compression_enabled
    and compression_device.startswith("cuda")
    and loader_workers > 0
):
    print(
        "Forcing num_workers=0 to avoid CUDA re-init in DataLoader workers during compression."
    )
    loader_workers = 0

codec = CompressAICodec(
    enabled=cfg.compression_enabled,
    model_name=cfg.compressai_model,
    quality=cfg.compressai_quality,
    pretrained=cfg.compression_pretrained,
    device=compression_device,
)

train_dataset = CompressedTrainingDataset(train_df, cfg, class_to_idx, codec)
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train_batch_size,
    shuffle=True,
    num_workers=loader_workers,
    pin_memory=torch.cuda.is_available(),
)
len(train_loader)

## Fine-tuning loop (compressed inputs)

In [ ]:
model = timm.create_model(
    cfg.backbone_name,
    pretrained=True,
    num_classes=num_classes,
    img_size=cfg.image_size,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate)

with mlflow.start_run(run_name="compression-baseline-train"):
    mlflow.log_params(
        {
            "image_size": cfg.image_size,
            "jpeg_normalize": cfg.jpeg_normalize,
            "jpeg_quality": cfg.jpeg_quality,
            "compression_enabled": cfg.compression_enabled,
            "compressai_model": cfg.compressai_model,
            "compressai_quality": cfg.compressai_quality,
            "backbone": cfg.backbone_name,
            "batch_size": cfg.train_batch_size,
            "epochs": cfg.epochs,
            "learning_rate": cfg.learning_rate,
        }
    )

    for epoch in range(cfg.epochs):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            running_correct += (preds == labels).sum().item()
            running_total += images.size(0)

        epoch_loss = running_loss / max(1, running_total)
        epoch_acc = running_correct / max(1, running_total)
        mlflow.log_metric("train/loss", epoch_loss, step=epoch)
        mlflow.log_metric("train/acc", epoch_acc, step=epoch)
        print(f"epoch={epoch} loss={epoch_loss:.4f} acc={epoch_acc:.4f}")

## Inference scaffold and submission file shape

In [ ]:
import csv

test_df = pd.read_csv(cfg.test_csv_path)
if "quadrat_id" not in test_df.columns:
    test_df["quadrat_id"] = test_df.iloc[:, 0]

# Placeholder inference output to keep workflow end-to-end executable.
# Replace `predictions` with tiled multi-label inference aggregation when calibrating thresholds.
submission = pd.DataFrame(
    {
        "quadrat_id": test_df["quadrat_id"],
        "species_ids": [[] for _ in range(len(test_df))],
    }
)
submission["species_ids"] = submission["species_ids"].apply(str)
submission.to_csv(cfg.submission_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved submission template to {cfg.submission_path}")